# Slot-count intervention in the identity format — all four conditions, eight runs, one Run all

**Question.** Five-slot identity training on k ∈ {0,2,3,4,5} generalized to singletons (notebook 10); five-slot
training on {0,2}, {0,2,3}, {0,2,3,4} did not (notebooks 12–14). Every training instruction in those three failing
supports contains at least one `<identity>` (I); train02345 also contains k=5 instructions with no I. This notebook
changes the number of physical slots S while keeping each logical training support fixed:

| condition | S | train k | no-I training records | held-out tests |
|---|---|---|---|---|
| slots2_train02 | 2 | 0, 2 | 640,000 (k=2) | k=1 at both positions |
| slots3_train023 | 3 | 0, 2, 3 | 426,666 (k=3) | k=1 at all 3 positions |
| slots4_train0234 | 4 | 0, 2, 3, 4 | 320,000 (k=4) | k=1 at all 4 positions |
| slots6_train02345 | 6 | 0, 2, 3, 4, 5 | 0 | k=1 at all 6 positions; all 729 six-operation words |

**Format.**
* Every instruction has exactly S VISIBLE tokens: the k A/B/C tokens fill a uniformly random ordered k-subset of the
  slots, and the rest are the learnable `<identity>` token. There is no PAD and no hidden slot.
* Each sequence has 77+S tokens with contiguous learned positions, and the shared position table has 83 rows.

**Recipe.** Notebooks 10/12–14 unchanged:
* Transformer AR and Looped AR, AdamW 3e-4, 5000 updates × 256, seed 42.
* Per-k allocation as in notebooks 12–14.
* For supports 02/023/0234, the logical (word, grid, target) stream and batch order are identical to the S=5 runs;
  only the layout changes. 02345 is a new stream, since notebook 10 used a different sampler.
* Common fresh initialization: the historical S=5 init with one added position row.
* Selection uses validation answer-digit CE (EOS excluded), macro-averaged over the trained k.

**Not a pure identity-token ablation.**
* Changing S also changes sequence length, downstream absolute positions, layouts, filler counts and compute.
* S=2/train02 keeps the even-count H ambiguity.
* Six-operation failure at S=6 confounds a larger k with the first-ever zero-I inputs.

One seed: descriptive only.

**Workflow.**
1. Run all: setup → shared audits → the sequential queue of eight runs (train → evaluate → diagnose → export each) →
   consolidated comparison.
2. If interrupted, Run all again: verified stages are skipped, training resumes from `state.pt`, and missing
   evaluation chunks are recomputed.
3. After any partial progress, the comparison cells can be rerun alone; they never train or build models.


In [ ]:
# --- 2. Bootstrap: locate or clone the PRIVATE repository (credentials are never printed or stored) ---
import base64, importlib.util, os, pathlib, subprocess, sys

REPO_URL = "https://github.com/zhexuandliu/Decompositionality.git"
REPO_BRANCH = "main"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

def _find_repo():
    candidates = [pathlib.Path("/content/Decompositionality"), pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    for p in candidates:
        if (p / "src" / "decompositionality").is_dir():
            return p.resolve()
    return None

REPO_DIR = _find_repo()
if REPO_DIR is None:
    token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
    if not token:
        import getpass
        token = getpass.getpass("GitHub token with read access to the private repo (input hidden): ")
    # Pass the credential through environment-scoped git config (not the URL, not the command line,
    # not persisted in .git/config).
    _hdr = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
    _env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_CONFIG_COUNT="1",
                GIT_CONFIG_KEY_0="http.https://github.com/.extraheader", GIT_CONFIG_VALUE_0=_hdr)
    REPO_DIR = pathlib.Path("/content/Decompositionality" if IN_COLAB else "Decompositionality").resolve()
    _r = subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
                        env=_env, capture_output=True, text=True)
    del token, _hdr, _env
    if _r.returncode != 0:
        raise RuntimeError("git clone failed (check the GITHUB_TOKEN secret / repo access):\n" + _r.stderr[-2000:])

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
for _mod in ("numpy", "pandas", "matplotlib", "torch"):
    if importlib.util.find_spec(_mod) is None:  # Colab already ships these; never reinstall torch/CUDA here
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _mod], check=True)
REPO_COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Repository:", REPO_DIR, "| commit:", REPO_COMMIT or "unknown")


In [ ]:
# --- CONFIGURATION (the only cell to edit) --------------------------------------------------------------------
MODE = os.environ.get("DECOMP_MODE", "full")   # "full" | "smoke" (tiny CPU integration check, separate paths, NOT results)
SEED = int(os.environ.get("DECOMP_SEED", 42))   # no automatic seed sweep
RUN_CONDITIONS = ["slots2_train02", "slots3_train023", "slots4_train0234", "slots6_train02345"]   # default: all four
RUN_MODELS = ["transformer_ar", "looped_transformer_ar"]                                          # default: both
RUN_STAGES = ["train", "evaluate", "diagnose", "export"]                                          # default: every stage
USE_DRIVE = IN_COLAB
RESULTS_ROOT = os.environ.get("DECOMP_RESULTS_ROOT") or (
    "/content/drive/MyDrive/decompositionality_results" if USE_DRIVE
    else "/content/decompositionality_results" if IN_COLAB else str(REPO_DIR / "results"))
SLOT_ROOT = f"{RESULTS_ROOT}/identity_ar_slot_count"
DEVICE = os.environ.get("DECOMP_DEVICE", "auto")
DTYPE = "auto"
MICROBATCH = None          # None = probe the largest safe microbatch (effective batch stays 256)
EVAL_CHUNK_GRIDS = 100     # 10 resumable chunks per run
# -----------------------------------------------------------------------------------------------------------------
if USE_DRIVE and IN_COLAB and RESULTS_ROOT.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
import json, pathlib
import pandas as pd
from IPython.display import Image, display
from decompositionality.config import make_experiment_config
from decompositionality import idslot as I, idslot_reports as R
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 250); pd.set_option("display.max_rows", 120)
FAMILY_ROOT = pathlib.Path(SLOT_ROOT) if MODE == "full" else pathlib.Path(SLOT_ROOT) / "smoke"
_c = make_experiment_config(RUN_CONDITIONS[0], MODE, SLOT_ROOT, RUN_MODELS, seed=SEED)
expected = dict(lr=3e-4, weight_decay=0.01, warmup_steps=250, min_lr=3e-5, grad_clip=1.0, val_every=250,
                optimizer_steps=5000, effective_batch_size=256)
print("resolved recipe:", _c.train.semantic_dict())
print("discrepancies from expected defaults:",
      {k: (getattr(_c.train, k), v) for k, v in expected.items() if getattr(_c.train, k) != v} or "none")
if MODE == "smoke":
    print("*** SMOKE MODE: tiny integration run under", FAMILY_ROOT, "- NOT benchmark results ***")
QUEUE = [(e, m) for e in R.CONDITIONS if e in RUN_CONDITIONS for m in R.MODELS if m in RUN_MODELS]
print(f"queue ({len(QUEUE)} runs, sequential):", " -> ".join(f"{e}/{m}" for e, m in QUEUE), "| stages:", RUN_STAGES)
print("results ->", FAMILY_ROOT)


In [ ]:
from decompositionality.utils import environment_info
print(json.dumps(environment_info(REPO_DIR), indent=1))


In [ ]:
# --- Shared audits (stop on failure): supports/counts, stream matching, layouts/masks/decoding, init, manifests -
AUD = FAMILY_ROOT / "audits" / f"seed_{SEED}"; AUD.mkdir(parents=True, exist_ok=True)
from decompositionality.data import load_or_build_manifests
man = load_or_build_manifests(FAMILY_ROOT / "manifests", SEED)
sup = R.support_audit(_c.train.effective_batch_size, _c.train.optimizer_steps)
stream, unif = R.stream_audit(SEED, _c.train.effective_batch_size, heldout=man.heldout_bytes)
lay = R.layout_audit(SEED)
ini = R.init_audit(SEED)
mb = R.microbatch_audit()
assert stream.deterministic_replay.all() and stream.only_training_k.all() and mb["equal_within_1e-6"]
assert (stream[stream.historical_S5_stream.str.startswith("idext")].same_words_grids_targets_order_as_S5 == True).all()
assert ini.identical_init_across_4_conditions.all() and ini.shared_tensors_equal_historical_S5_construction.all()
cfgs = {e: make_experiment_config(e, MODE, SLOT_ROOT, RUN_MODELS, seed=SEED) for e in R.CONDITIONS}
ctxs = {e: R.prepare_condition(c, str(REPO_DIR), {
            "five-slot count extrapolation (nb 12-14)": f"{RESULTS_ROOT}/identity_ar_count_extrapolation/manifests/manifest.json",
            "identity_02345 (nb 10)": f"{RESULTS_ROOT}/identity_varlen_02345_all_models/manifests/manifest.json"})
        for e, c in cfgs.items()}
tm = R.test_manifest_audit(ctxs)
for name, df in (("support", sup), ("stream_match", stream), ("placement_uniformity", unif), ("layout", lay),
                 ("initialization", ini), ("test_manifest", tm)):
    df.to_csv(AUD / f"{name}.csv", index=False); print(f"\n[{name}]"); display(df)
(AUD / "microbatch.json").write_text(json.dumps(mb, indent=1)); print("[microbatch]", mb)
print("training-transformation coverage:"); display(I.transformation_coverage())
del ctxs


In [ ]:
# --- Sequential eight-run driver (one model/optimizer on the GPU at a time; resumable; skips verified stages) ---
status = R.run_queue(SLOT_ROOT, MODE, SEED, RUN_CONDITIONS, RUN_MODELS, RUN_STAGES, repo_dir=str(REPO_DIR),
                     compare_manifests={"five-slot count extrapolation (nb 12-14)":
                                        f"{RESULTS_ROOT}/identity_ar_count_extrapolation/manifests/manifest.json"},
                     chunk_grids=EVAL_CHUNK_GRIDS, device=DEVICE, dtype=DTYPE, microbatch=MICROBATCH)
display(status)


In [ ]:
# --- Consolidated comparison (reads saved artifacts only; rerun alone at any time) -----------------------------
res = R.analyse(FAMILY_ROOT, RESULTS_ROOT, SEED)
FIG = res["out"] / "figures"
for p in R.figures(res, FIG):
    if str(p).endswith(".png"):
        display(Image(filename=str(p)))
print("(a) main table"); display(res["main"])
print("(c) accuracy by k"); display(res["by_k"])
print("(e) historical S=5 vs new S at the same support"); display(res["historical"]); display(res["historical_meta"])
display(res["historical_paired_seen"])
print("(f) six-operation per-word (S=6)"); display(res["six_op_per_word"].sort_values(["model", "exact_acc"]) if len(res["six_op_per_word"]) else "not yet available")
print("six-operation failure subsequence checks"); display(res["six_op_subsequence"])
print("(g) identity-count view"); display(res["identity_count_view"])
print("singleton consistency across placements"); display(res["singleton_consistency"])
print("(h) audits"); display(res["audit_initialization"]); display(res["timing"])


In [ ]:
# --- Export + eight-row stage status (a run is complete only when every stage is verified on disk) -------------
arch = R.export_comparisons(res)
st = R.status_table(FAMILY_ROOT, SEED); display(st)
sel = st[st.condition.isin(RUN_CONDITIONS) & st.model.isin(RUN_MODELS)]
print(f"{int(sel.complete.sum())}/{len(sel)} selected runs complete" +
      ("" if sel.complete.all() else " -> rerun Run all to resume the rest"))
